# ETL Pipeline Log Analysis with OpenSearch + Amazon Bedrock

This notebook lets you query your MWAA ETL pipeline logs using **natural language**.

It works by:
1. Taking your plain English question
2. Using **Claude 3 Haiku** (via Amazon Bedrock) to translate it into an OpenSearch DSL query
3. Executing the query against your `etl-logs-*` index
4. Using Claude to summarize the results in plain English

**Index:** `etl-logs-2026-04`  
**OpenSearch Domain:** `etl-monitoring`  
**Region:** `us-east-1`

## 1. Install Dependencies

In [ ]:
!pip install requests requests-aws4auth boto3 -q

## 2. Configuration

In [ ]:
import boto3
import json
import requests
from requests_aws4auth import AWS4Auth
from datetime import datetime, timezone

# OpenSearch configuration
OPENSEARCH_ENDPOINT = "https://search-etl-monitoring-mz7zpvh76up33iejfbl7ock3oq.us-east-1.es.amazonaws.com"
INDEX_NAME = "etl-logs-2026-04"
REGION = "us-east-1"

# Bedrock model
BEDROCK_MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"

# Build SigV4 auth for OpenSearch
session = boto3.session.Session()
credentials = session.get_credentials().get_frozen_credentials()
auth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    REGION,
    "es",
    session_token=credentials.token
)

# Bedrock client
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print("Configuration loaded successfully")

## 3. Helper Functions

In [ ]:
# Log_Event schema for context
SCHEMA_DESCRIPTION = """
The OpenSearch index 'etl-logs-2026-04' contains Log_Event documents from an MWAA ETL pipeline.

Fields:
- run_id (keyword): MWAA DAG run ID, e.g. 'manual__2026-04-24T14:16:54.328819+00:00'
- task_name (keyword): Airflow task ID, e.g. 'glue_extraction', 'lambda_transform', 'ec2_custom_script'
- component_type (keyword): 'glue', 'lambda', 'ec2', or 'mwaa'
- log_level (keyword): 'INFO', 'WARN', or 'ERROR'
- message (text): Human-readable description of the event
- timestamp (date): ISO 8601 UTC, e.g. '2026-04-24T14:17:19Z'
- duration_ms (long): Task duration in milliseconds
- start_time (date): Task start time
- end_time (date): Task end time
- function_name (keyword): Lambda function name (lambda only)
- request_id (keyword): Lambda request ID (lambda only)
- outcome (keyword): 'success' or 'error' (lambda only)
- instance_id (keyword): EC2 instance ID (ec2 only)
- script_name (keyword): Script filename (ec2 only)
- exit_code (integer): Script exit code (ec2 only)
- job_name (keyword): Glue job name (glue only)
- job_run_id (keyword): Glue job run ID (glue only)
- terminal_status (keyword): 'SUCCEEDED', 'FAILED', 'STOPPED' (glue only)
"""


def query_opensearch(dsl_query: dict) -> dict:
    """Execute a DSL query against OpenSearch and return the results."""
    url = f"{OPENSEARCH_ENDPOINT}/{INDEX_NAME}/_search"
    response = requests.post(
        url,
        data=json.dumps(dsl_query),
        auth=auth,
        headers={"Content-Type": "application/json"}
    )
    response.raise_for_status()
    return response.json()


def ask_claude(prompt: str) -> str:
    """Send a prompt to Claude 3 Haiku via Bedrock and return the response."""
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 2048,
        "messages": [{"role": "user", "content": prompt}]
    })
    response = bedrock.invoke_model(modelId=BEDROCK_MODEL_ID, body=body)
    result = json.loads(response["body"].read())
    return result["content"][0]["text"]


def natural_language_query(question: str, verbose: bool = False) -> str:
    """
    Answer a natural language question about your ETL pipeline logs.
    
    1. Uses Claude to translate the question into a DSL query
    2. Executes the query against OpenSearch
    3. Uses Claude to summarize the results
    """
    # Step 1: Translate question to DSL
    translation_prompt = f"""
You are an OpenSearch DSL expert. Convert the following natural language question into a valid OpenSearch DSL query JSON.

Schema context:
{SCHEMA_DESCRIPTION}

Rules:
- Return ONLY valid JSON, no explanation
- Use 'keyword' fields for exact matches (term/terms queries)
- Use 'text' fields for full-text search (match queries)
- For time ranges, use ISO 8601 UTC format
- For aggregations, set size: 0
- Default size: 10 unless the question asks for more
- Always sort by timestamp descending unless otherwise specified
- For 'most recent' or 'latest', sort by timestamp desc and size 1 or the run_id of the latest event

Question: {question}

DSL Query JSON:
"""
    
    dsl_text = ask_claude(translation_prompt).strip()
    
    # Extract JSON from the response
    if "```json" in dsl_text:
        dsl_text = dsl_text.split("```json")[1].split("```")[0].strip()
    elif "```" in dsl_text:
        dsl_text = dsl_text.split("```")[1].split("```")[0].strip()
    
    dsl_query = json.loads(dsl_text)
    
    if verbose:
        print("Generated DSL Query:")
        print(json.dumps(dsl_query, indent=2))
        print()
    
    # Step 2: Execute the query
    results = query_opensearch(dsl_query)
    
    if verbose:
        print(f"Raw results: {results['hits']['total']['value']} documents found")
        print()
    
    # Step 3: Summarize results
    summary_prompt = f"""
You are an ETL pipeline observability assistant. A user asked the following question about their MWAA pipeline logs:

Question: {question}

Here are the OpenSearch query results:
{json.dumps(results, indent=2)[:4000]}

Provide a clear, concise answer to the question based on the data. 
- Highlight any errors, failures, or anomalies
- Include specific values (run_ids, durations, timestamps) where relevant
- If no data was found, say so clearly
- Keep the response focused and actionable
"""
    
    return ask_claude(summary_prompt)


print("Helper functions loaded. Ready to query!")

## 4. Natural Language Queries

Ask any question about your ETL pipeline logs in plain English.

In [ ]:
# Show the generated DSL query alongside the answer
answer = natural_language_query(
    "What insights can be generated from my most recent DAG run?",
    verbose=True
)
print(answer)

In [ ]:
print(natural_language_query("Show me all ERROR events from the last 24 hours"))

In [ ]:
print(natural_language_query("What is the average duration for each component type?"))

In [ ]:
print(natural_language_query("How many successful pipeline runs have there been?"))

In [ ]:
print(natural_language_query("Which tasks have failed and what were the error messages?"))

## 5. Interactive Query Cell

Change the question below and re-run the cell to ask anything.

In [ ]:
# Change this question and run the cell
MY_QUESTION = "Show me all events from the most recent successful run"

print(f"Question: {MY_QUESTION}")
print("-" * 60)
print(natural_language_query(MY_QUESTION))

## 6. Direct DSL Queries

You can also run DSL queries directly if you prefer.

In [ ]:
# Direct DSL: all events for the most recent run
results = query_opensearch({
    "query": {"match_all": {}},
    "size": 1,
    "sort": [{"timestamp": {"order": "desc"}}]
})

latest_run_id = results["hits"]["hits"][0]["_source"]["run_id"]
print(f"Latest run_id: {latest_run_id}")

# Get all events for that run
run_events = query_opensearch({
    "query": {"term": {"run_id": latest_run_id}},
    "size": 20,
    "sort": [{"timestamp": {"order": "asc"}}]
})

print(f"\nEvents for run {latest_run_id}:")
for hit in run_events["hits"]["hits"]:
    s = hit["_source"]
    print(f"  [{s['timestamp'][:19]}] {s['component_type']:6} {s['log_level']:5} {s['task_name']}")
    print(f"    {s['message'][:80]}")

In [ ]:
# Aggregation: event count by log level
agg_results = query_opensearch({
    "size": 0,
    "aggs": {
        "by_log_level": {
            "terms": {"field": "log_level"}
        }
    }
})

print("Event count by log level:")
for bucket in agg_results["aggregations"]["by_log_level"]["buckets"]:
    print(f"  {bucket['key']:6}: {bucket['doc_count']} events")

In [ ]:
# Aggregation: average duration by component type
duration_results = query_opensearch({
    "size": 0,
    "aggs": {
        "by_component": {
            "terms": {"field": "component_type"},
            "aggs": {
                "avg_duration_ms": {"avg": {"field": "duration_ms"}},
                "max_duration_ms": {"max": {"field": "duration_ms"}}
            }
        }
    }
})

print("Duration stats by component type:")
for bucket in duration_results["aggregations"]["by_component"]["buckets"]:
    avg = bucket["avg_duration_ms"]["value"]
    max_d = bucket["max_duration_ms"]["value"]
    print(f"  {bucket['key']:8}: avg={avg:.0f}ms  max={max_d:.0f}ms  count={bucket['doc_count']}")